# HDB Rentals: progress-report pipeline

This notebook reproduces the progress report's findings as of the checkpoint (slides 4 to 13), in one place
that runs on this branch alone, and then does next steps 3 and 4: combine location features with the
market-level model, and write a first submission. Nothing here goes beyond the report; work done after the
checkpoint (nine-origin level check, model comparison, tuning) is in `08_level_robustness.ipynb`.

| section | slide | source |
|---|---|---|
| 1. Clean the rental data | 4 | `main.ipynb` §1 (summary in cell 44), `01_eda_train` §4 |
| 2. Block-level location features | 5, 12 | `01_eda_hdb`/`mrt`/`school`, `06_distance_features` §2, §4e (`auxiliary-distance-features`) |
| 3. Validation: five walk-forward folds + one 16-month fold | 8 | `macro-time-trend`: `src/hdb_rentals/validation.py` |
| 4. Baseline and the horizon problem | 9 | `macro-time-trend`: `baselines.py`, `experiments.jsonl` |
| 5. Location features | 12 | `06_distance_features` §4e |
| 6. Market level × flat premium | 11 | `macro-time-trend`: `market_level.py` |
| 7. Macro data | 10 | `macro-time-trend`: `macro.py` (not re-run) |
| 8. Combined model | next step 3 | this notebook |
| 9. Fit on all of train, first submission | next step 4 | this notebook |
| 10. Summary | 13 | |

The `macro-time-trend` helpers are rewritten inline (same logic, no `src` import), so the notebook needs only
`data/`, `cleaned/` and the auxiliary files.

In [1]:
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA, CLEANED, OUT = ROOT / "data", ROOT / "cleaned", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "MONTHLY_RENT"
KEY = ["TOWN", "BLOCK", "STREET"]
SEED = 0
T0 = time.time()

## 1. Clean the rental data (slide 4)

Train and test go through the same function:

- lowercase and strip every text column; "4 room" and "4-room" become one level, so 11 flat-type labels collapse
  to 6, ordinal-encoded 1 to 6 as `FLAT_TYPE_ORD`;
- drop the constant `FURNISHED` and `FEE`;
- add `FLAT_AGE` (approval year minus lease start) and the month counter `MONTH_INDEX` (2021-01 = 0);
- `STREET` is kept and `BLOCK` dropped from the model inputs (576 streets vs 9,406 block + street pairs). `BLOCK`
  is still used as part of the join key in section 2.

Train is sorted by month (stable), as `data.load_train` does on `macro-time-trend`, so rows reach LightGBM in the
same order as in the logged runs. Test keeps the file's order, since the submission is matched by row position.

In [2]:
FLAT_TYPE_ORDER = ["1-room", "2-room", "3-room", "4-room", "5-room", "executive"]


def lower_strip(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df: pd.DataFrame) -> pd.DataFrame:
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    # "4 room" and "4-room" are the same level.
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    date = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = date
    out["MONTH_INDEX"] = (date.dt.year - 2021) * 12 + date.dt.month - 1
    out["FLAT_AGE"] = date.dt.year - out["LEASE_COMMENCE_DATE"]
    out["FLAT_TYPE_ORD"] = out["FLAT_TYPE"].map({t: i + 1 for i, t in enumerate(FLAT_TYPE_ORDER)})
    return out


raw_train, raw_test = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
train = clean_rent(raw_train).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
test = clean_rent(raw_test)
assert train["FLAT_TYPE_ORD"].notna().all() and test["FLAT_TYPE_ORD"].notna().all()

print("flat-type labels before cleaning:", raw_train["FLAT_TYPE"].nunique(), "->", train["FLAT_TYPE"].nunique())
print("streets:", train["STREET"].nunique(), " block + street pairs:", train[["BLOCK", "STREET"]].drop_duplicates().shape[0])
for name, df in [("train", train), ("test", test)]:
    print(f"{name:<5} {len(df):>7,} rows  {df['RENT_APPROVAL_DATE'].min():%Y-%m} to {df['RENT_APPROVAL_DATE'].max():%Y-%m}")

flat-type labels before cleaning: 11 -> 6
streets: 576  block + street pairs: 9406
train 150,000 rows  2021-01 to 2025-03
test   50,000 rows  2025-03 to 2026-07


## 2. Block-level location features (slides 5, 12)

Built as in `06_distance_features` §2 from the cleaned tables in `cleaned/` (slide 5: lowercased join key, 11
blocks' `-1` max floor / year completed set to missing, 8 school coordinates corrected) plus the raw mall and
hawker files. Planned MRT stations and hawker centres under construction are excluded. CBD = Raffles Place MRT.

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))

with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
    features = json.load(f)["features"]
hawker = lower_strip(pd.DataFrame({
    "NAME": [feat["properties"]["NAME"] for feat in features],
    "STATUS": [feat["properties"]["STATUS"] for feat in features],
    "LONGITUDE": [feat["geometry"]["coordinates"][0] for feat in features],
    "LATITUDE": [feat["geometry"]["coordinates"][1] for feat in features],
}))

mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
POI_GROUPS = {
    "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
    "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
    "mrt": mrt_open,
    "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
    "malls": malls,
    "hawker": hawker[hawker["STATUS"].ne("under construction")],
}
RADII_KM = {"1km": 1.0, "2km": 2.0}

lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
for prefix, poi in POI_GROUPS.items():
    d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
    hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
    for label, r in RADII_KM.items():
        hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])

print("hawker centres:", len(hawker), "->", len(POI_GROUPS["hawker"]), "after dropping those under construction")
print({k: len(v) for k, v in POI_GROUPS.items()})

hawker centres: 129 -> 123 after dropping those under construction
{'primary': 179, 'secondary': 133, 'mrt': 192, 'mrt_interchange': 44, 'malls': 113, 'hawker': 123}


`DIST` is the stacked set behind slide 12's 496.4 → 489.6: `06_distance_features` §4e, "2km REFERENCE + kitchen
sink". It mixes counts within 1 and 2 km with nearest distances, plus `MAX_FLOOR` and distance to the CBD.

In [4]:
DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]

block_cols = KEY + ["MISSING_BUILDING_INFO"] + DIST
train = train.merge(hdb[block_cols], on=KEY, how="left", validate="m:1")
test = test.merge(hdb[block_cols], on=KEY, how="left", validate="m:1")
for name, df in [("train", train), ("test", test)]:
    print(f"{name:<5} rows matched to a block: {df['MISSING_BUILDING_INFO'].notna().mean():.2%}  "
          f"SERS rows (MAX_FLOOR missing): {int(df['MISSING_BUILDING_INFO'].eq(True).sum())}")
train[DIST].describe().T[["mean", "min", "50%", "max"]]

train rows matched to a block: 100.00%  SERS rows (MAX_FLOOR missing): 159
test  rows matched to a block: 100.00%  SERS rows (MAX_FLOOR missing): 0


,mean,min,50%,max
MAX_FLOOR,15.30,2.00,13.00,50.00
primary_within_2km,8.11,0.00,7.00,20.00
mrt_within_2km,5.60,0.00,5.00,23.00
primary_nearest_km,0.43,0.02,0.37,3.31
mrt_nearest_km,0.59,0.02,0.53,2.57
secondary_within_1km,2.05,0.00,2.00,6.00
mrt_interchange_within_1km,0.45,0.00,0.00,4.00
mrt_interchange_nearest_km,1.50,0.03,1.15,6.69
malls_within_1km,1.17,0.00,1.00,12.00
malls_nearest_km,0.81,0.00,0.73,3.20


## 3. Validation (slide 8)

Split by time only. Two schemes, rewritten from `validation.py`:

- **Five walk-forward folds** (`expanding_window_splits`): each trains on every month before a cutoff and scores
  the next 3. The last fold scores 2025-01 to 2025-03. The headline is the mean, but folds are compared one by one.
- **One 16-month fold** (`horizon_matched_split`): train through 2023-11, score 2023-12 to 2025-03, the same
  distance ahead as the test window.
- **`horizon_report`** breaks the 16-month fold's error down by months ahead (1-3, 4-6, 7-9, 10-12, 13-16). Bias is
  mean(prediction − actual), so negative is under-prediction.

In [5]:
def expanding_window_splits(df, n_splits=5, valid_months=3):
    months = df["MONTH_INDEX"].to_numpy()
    unique = np.sort(np.unique(months))
    for k in range(n_splits, 0, -1):
        v_end = len(unique) - (k - 1) * valid_months
        v_start = v_end - valid_months
        yield np.flatnonzero(months < unique[v_start]), np.flatnonzero(np.isin(months, unique[v_start:v_end]))


def horizon_matched_split(df, horizon_months=16):
    months = df["MONTH_INDEX"].to_numpy()
    cutoff = months.max() - horizon_months + 1
    return np.flatnonzero(months < cutoff), np.flatnonzero(months >= cutoff)


def rmse(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y, dtype=float) - np.asarray(pred, dtype=float)) ** 2)))


def bias(y, pred):
    return float(np.mean(np.asarray(pred, dtype=float) - np.asarray(y, dtype=float)))


def horizon_report(df, y_true, y_pred, trn, val, buckets=((1, 3), (4, 6), (7, 9), (10, 12), (13, 16))):
    months = df["MONTH_INDEX"].to_numpy()
    ahead = months[val] - months[trn].max()
    t, p = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    rows = []
    for lo, hi in buckets:
        sel = (ahead >= lo) & (ahead <= hi)
        rows.append({"months ahead": f"{lo}-{hi}", "n": int(sel.sum()), "rmse": rmse(t[sel], p[sel]), "bias": bias(t[sel], p[sel])})
    return pd.DataFrame(rows).set_index("months ahead")


month_label = lambda m: f"{2021 + m // 12}-{m % 12 + 1:02d}"
FOLDS = list(expanding_window_splits(train)) + [horizon_matched_split(train)]
FOLD_NAMES = [f"fold {i}" for i in range(1, 6)] + ["16-month"]
months = train["MONTH_INDEX"].to_numpy()
pd.DataFrame([
    {"fold": name, "train": f"2021-01 to {month_label(months[t].max())}", "scored": f"{month_label(months[v].min())} to {month_label(months[v].max())}",
     "train rows": len(t), "scored rows": len(v)}
    for name, (t, v) in zip(FOLD_NAMES, FOLDS)
]).set_index("fold")

,train,scored,train rows,scored rows
fold,,,,
fold 1,2021-01 to 2023-12,2024-01 to 2024-03,105837,9005
fold 2,2021-01 to 2024-03,2024-04 to 2024-06,114842,9174
fold 3,2021-01 to 2024-06,2024-07 to 2024-09,124016,8786
fold 4,2021-01 to 2024-09,2024-10 to 2024-12,132802,8274
fold 5,2021-01 to 2024-12,2025-01 to 2025-03,141076,8924
16-month,2021-01 to 2023-11,2023-12 to 2025-03,102829,47171


## 4. Baseline and the horizon problem (slide 9)

LightGBM on the six raw columns with the logged settings (`n_estimators=400, learning_rate=0.05, num_leaves=63`,
LightGBM defaults otherwise). Text columns are pandas categories with the level set fixed once from all of train,
as `baselines.model_matrix` does, so train, folds and test share one code per level. Report: five folds 540.4,
517.6, 484.3, 458.2, 466.5 (mean 493.4); 16-month fold 506.2; bias −7 at 1-3 months ahead, −80 at 13-16.

In [6]:
RAW_FEATURES = ["FLAT_AGE", "FLAT_MODEL", "FLAT_TYPE", "FLOOR_AREA_SQM", "MONTH_INDEX", "TOWN"]
LGBM_PARAMS = {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63}
CATEGORIES = {c: pd.CategoricalDtype(sorted(train[c].dropna().unique())) for c in ["FLAT_MODEL", "FLAT_TYPE", "TOWN", "STREET"]}


def model_matrix(df, features):
    # Fixed level sets from train; a level train never had becomes missing.
    X = df[features].copy()
    for c in features:
        if c in CATEGORIES:
            X[c] = X[c].astype(CATEGORIES[c])
    return X


def lgbm(X, y):
    return lgb.LGBMRegressor(**LGBM_PARAMS, verbose=-1).fit(X, y)


def run_folds(predict, label):
    # predict(trn, val, df) -> predictions for df.iloc[val]. Scores the five folds and the 16-month fold.
    y = train[TARGET].to_numpy()
    rows, horizon = [], None
    for name, (t, v) in zip(FOLD_NAMES, FOLDS):
        pred = predict(train, t, v)
        rows.append({"run": label, "fold": name, "rmse": rmse(y[v], pred), "bias": bias(y[v], pred)})
        if name == "16-month":
            horizon = horizon_report(train, y[v], pred, t, v)
    return pd.DataFrame(rows), horizon


def fold_table(results):
    rm = results.pivot(index="run", columns="fold", values="rmse")[FOLD_NAMES]
    rm.insert(5, "5-fold mean", rm[FOLD_NAMES[:5]].mean(axis=1))
    return rm.loc[results["run"].unique()]


def raw_predict(features):
    def predict(df, t, v):
        X, y = model_matrix(df, features), df[TARGET].to_numpy()
        return lgbm(X.iloc[t], y[t]).predict(X.iloc[v])
    return predict


base_results, base_horizon = run_folds(raw_predict(RAW_FEATURES), "baseline (6 raw columns)")
display(fold_table(base_results))
base_results.pivot(index="run", columns="fold", values="bias")[FOLD_NAMES]

fold,fold 1,fold 2,fold 3,fold 4,fold 5,5-fold mean,16-month
run,,,,,,,
baseline (6 raw columns),540.36,517.64,484.31,458.21,466.53,493.41,506.18


fold,fold 1,fold 2,fold 3,fold 4,fold 5,16-month
run,,,,,,
baseline (6 raw columns),-1.98,-2.93,-35.22,-31.35,-4.28,-50.00


In [7]:
base_horizon

,n,rmse,bias
months ahead,,,
1-3,8829,544.66,-7.21
4-6,9347,526.12,-24.56
7-9,9089,506.24,-63.22
10-12,8086,480.11,-67.27
13-16,11820,476.74,-80.08


- **Reproduces slide 9 exactly:** folds 540.4, 517.6, 484.3, 458.2, 466.5, mean 493.4; 16-month fold 506.2.
- **Fold spread is 82** (540.4 to 458.2), larger than any feature gain in this notebook, so later runs are
  compared fold by fold.
- **Under-prediction grows with the horizon:** bias −7.2 at 1-3 months ahead, −24.6, −63.2, −67.3, then −80.1 at
  13-16. RMSE falls over the same range (544.7 → 476.7), so RMSE alone hides it. The tree holds the level of its
  last training month.

## 5. Location features (slide 12)

**Reproduction on `06`'s single split** (fit 2021-01 to 2023-12, score 2024-01 to 2025-03), with `06`'s own
columns and settings (`num_leaves=31`, categories from the fit rows): BASE should give ≈ 496.4 and BASE + `DIST`
≈ 489.6.

In [8]:
def fit_predict_06(num, cat, fit_df, apply_df):
    Xf, Xa = fit_df[num + cat].copy(), apply_df[num + cat].copy()
    for c in cat:
        Xf[c] = Xf[c].astype("category")
        Xa[c] = pd.Categorical(Xa[c].where(Xa[c].isin(Xf[c].cat.categories)), categories=Xf[c].cat.categories)
    est = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, verbose=-1, n_jobs=-1, random_state=SEED)
    return est.fit(Xf, fit_df[TARGET], categorical_feature=cat).predict(Xa)


NUM_06, CAT_06 = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"], ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
cut = pd.Timestamp("2024-01-01")
fit_06, val_06 = train[train["RENT_APPROVAL_DATE"] < cut], train[train["RENT_APPROVAL_DATE"] >= cut]
y_06 = val_06[TARGET].to_numpy()
pd.DataFrame({
    name: {"rmse": rmse(y_06, p), "bias": bias(y_06, p)}
    for name, p in [("BASE", fit_predict_06(NUM_06, CAT_06, fit_06, val_06)),
                    ("BASE + DIST", fit_predict_06(NUM_06 + DIST, CAT_06, fit_06, val_06))]
}).T

,rmse,bias
BASE,496.37,-35.98
BASE + DIST,489.58,-38.11


**Same features on the slide-8 folds.** The baseline's six raw columns plus all of `DIST`, baseline settings. A
plain result: no per-feature selection, which is next step 2 and not yet done.

In [9]:
dist_results, dist_horizon = run_folds(raw_predict(RAW_FEATURES + DIST), "baseline + DIST")
loc_results = pd.concat([base_results, dist_results])
display(fold_table(loc_results))
loc_results.pivot(index="run", columns="fold", values="bias").loc[loc_results["run"].unique(), FOLD_NAMES]

fold,fold 1,fold 2,fold 3,fold 4,fold 5,5-fold mean,16-month
run,,,,,,,
baseline (6 raw columns),540.36,517.64,484.31,458.21,466.53,493.41,506.18
baseline + DIST,533.20,511.14,479.63,450.62,452.87,485.49,494.89


fold,fold 1,fold 2,fold 3,fold 4,fold 5,16-month
run,,,,,,
baseline (6 raw columns),-1.98,-2.93,-35.22,-31.35,-4.28,-50.00
baseline + DIST,-2.27,-6.65,-39.41,-25.88,-6.18,-47.40


- **Reproduces slide 12:** BASE 496.4 → BASE + `DIST` 489.6 (+6.8) on `06`'s single split.
- **On the slide-8 folds, `DIST` helps on every fold:** 540.4 → 533.2, 517.6 → 511.1, 484.3 → 479.6,
  458.2 → 450.6, 466.5 → 452.9 (mean 493.4 → 485.5); 16-month fold 506.2 → 494.9.
- Gains per fold (4.7 to 13.7) are smaller than the spread across folds (82), which is why they are read fold by
  fold. Bias barely moves (16-month −50.0 → −47.4): location does not fix the level.
- This is all 13 `DIST` columns at once. Which ones beat fold noise on their own is next step 2, not done here.

## 6. Market level × flat premium (slide 11)

Following `market_level.py`:

  rent ≈ market level(month) × premium(flat)

- **Level:** a hedonic index, the month coefficients of an OLS of log rent on month dummies plus `TOWN`,
  `FLAT_TYPE`, `FLAT_MODEL` (dummies), `FLOOR_AREA_SQM` and `FLAT_AGE`. Fitted on training months only.
- **Premium:** LightGBM with the baseline settings learns rent ÷ level on the five raw columns other than
  `MONTH_INDEX`; prediction = premium × forecast level.
- **Level forecasts** past the last training month:
  - frozen: the current baseline (section 4), a tree on `MONTH_INDEX` that holds the last month's level;
  - damped trend: Holt's additive trend with damping (`statsmodels`, estimated initial values);
  - straight line: a line fitted through all training months' log level, continued from the last month;
  - perfect level: the level fitted with the scored months' own rents too. Sees the answers; an upper bound.
- **Two cutoffs, each scored on the next 16 months:** plateau (train to 2023-11, the 16-month fold from section
  3) and climb (train to 2022-06, score 2022-07 to 2023-10).

In [10]:
HEDONIC_CAT, HEDONIC_NUM = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"], ["FLOOR_AREA_SQM", "FLAT_AGE"]


def hedonic_level(df, idx):
    # Log market level per MONTH_INDEX from the rows in idx, relative to the first month.
    rows = df.iloc[idx]
    months = rows["MONTH_INDEX"]
    X = pd.concat([
        pd.get_dummies(months, prefix="m", drop_first=True, dtype=float),
        pd.get_dummies(rows[HEDONIC_CAT], drop_first=True, dtype=float),
        rows[HEDONIC_NUM].astype(float),
    ], axis=1)
    X.insert(0, "const", 1.0)
    coef, *_ = np.linalg.lstsq(X.to_numpy(), np.log(rows[TARGET].to_numpy()), rcond=None)
    coef = pd.Series(coef, index=X.columns)
    ordered = np.sort(months.unique())
    return pd.Series([0.0] + [coef[f"m_{m}"] for m in ordered[1:]], index=ordered)


def forecast_level(level, months, method):
    # Carry a training-months log level forward to the months past its end.
    ahead = np.sort(np.unique(months[months > level.index.max()]))
    if method == "damped":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # statsmodels convergence warnings; the fit is unchanged
            fit = ExponentialSmoothing(level.to_numpy(), trend="add", damped_trend=True,
                                       initialization_method="estimated").fit()
        future = fit.forecast(len(ahead))
    elif method == "linear":
        slope, _ = np.polyfit(level.index, level.to_numpy(), 1)
        future = level.iloc[-1] + slope * np.arange(1, len(ahead) + 1)
    elif method == "hold":
        future = np.full(len(ahead), level.iloc[-1])
    else:
        raise ValueError(method)
    return pd.concat([level, pd.Series(future, index=ahead)])


def level_scale(df, t, v, method):
    # Market level (not log) for every row of df; v rows get the forecast.
    months = df["MONTH_INDEX"].to_numpy()
    if method == "oracle":
        level = hedonic_level(df, np.concatenate([t, v]))
    else:
        level = forecast_level(hedonic_level(df, t), months[v], method)
    return np.exp(level.reindex(months).to_numpy())


def level_predict(features, method):
    def predict(df, t, v):
        X, y = model_matrix(df, features), df[TARGET].to_numpy()
        scale = level_scale(df, t, v, method)
        return lgbm(X.iloc[t], y[t] / scale[t]).predict(X.iloc[v]) * scale[v]
    return predict


PREMIUM_FEATURES = [c for c in RAW_FEATURES if c != "MONTH_INDEX"]
ARMS = {
    "frozen (current baseline)": raw_predict(RAW_FEATURES),
    "damped trend": level_predict(PREMIUM_FEATURES, "damped"),
    "straight line": level_predict(PREMIUM_FEATURES, "linear"),
    "perfect level (upper bound)": level_predict(PREMIUM_FEATURES, "oracle"),
}
CUTOFFS = {"plateau (to 2023-11)": train, "climb (to 2022-06)": train[train["MONTH_INDEX"] <= 33].reset_index(drop=True)}

level_rows = []
for cutoff, df in CUTOFFS.items():
    t, v = horizon_matched_split(df)
    y = df[TARGET].to_numpy()
    for arm, predict in ARMS.items():
        pred = predict(df, t, v)
        h = horizon_report(df, y[v], pred, t, v)
        level_rows.append({"forecast": arm, "cutoff": cutoff, "rmse": rmse(y[v], pred), "bias 13-16": h.loc["13-16", "bias"]})
level_results = pd.DataFrame(level_rows)
level_results.pivot(index="forecast", columns="cutoff", values=["rmse", "bias 13-16"]).loc[list(ARMS)]

rmse                              bias 13-16                     
cutoff                      climb (to 2022-06) plateau (to 2023-11) climb (to 2022-06) plateau (to 2023-11)
forecast                                                                                                   
frozen (current baseline)               793.34               506.18            -619.65               -80.08
damped trend                            620.01               500.11              86.49               -49.86
straight line                           661.91               656.20            -242.50               651.79
perfect level (upper bound)             610.99               499.90             -12.55                41.24

- **Reproduces slide 11 exactly:** frozen 506.2 / 793.3, damped 500.1 / 620.0, straight line 656.2 / 661.9,
  perfect level 499.9 / 611.0 (plateau / climb).
- **Damped trend is close to the upper bound at both cutoffs:** 0.2 above it on the plateau, 9.0 above it in the
  climb. It cuts 13-16-month bias from −80.1 to −49.9 on the plateau and from −619.7 to +86.5 in the climb.
- **The straight line overshoots on the plateau** (+651.8 bias at 13-16 months) because its slope averages in the
  2021-23 climb.

**Caveat:** a wider check after the report (nine cutoffs, see `08_level_robustness.ipynb`) found the damped trend
over-predicts badly from a 2022-12 cutoff (bias about +533, RMSE about 871), so this two-cutoff result is not
conclusive.

## 7. Macro data (slide 10)

Not used. On the `macro-time-trend` branch (`src/hdb_rentals/macro.py`, `experiments.jsonl`), COE prices did no
better than an unrelated share price used as a placebo (504.4 vs 505.6 vs 506.2 on the 16-month fold), so macro
features were dropped. Not re-run here.

## 8. Combined model (next step 3)

Location features in the premium model, damped-trend level on top:

- **Premium columns** are slide 4's: `FLAT_TYPE_ORD` (1 to 6), `FLAT_AGE`, `TOWN`, `FLAT_MODEL`, `FLOOR_AREA_SQM`
  (`main.ipynb` keeps it "for now"), `STREET`, no `BLOCK`, plus all of `DIST`. `main.ipynb` one-hot encodes
  `STREET`; here it is a LightGBM native categorical (576 levels), with the level set fixed from all of train.
  No `MONTH_INDEX`: time enters only through the level.
- **Level:** hedonic index, damped-trend forecast, as in section 6.
- Baseline settings for LightGBM. Scored on the slide-8 folds next to the baseline, with the damped level on
  the raw columns (section 6's model) as the step in between.

In [11]:
SLIDE4_FEATURES = ["FLAT_TYPE_ORD", "FLAT_AGE", "TOWN", "FLAT_MODEL", "FLOOR_AREA_SQM", "STREET"]
COMBINED_FEATURES = SLIDE4_FEATURES + DIST
LEVEL_METHOD = "damped"

damped_raw_results, _ = run_folds(level_predict(PREMIUM_FEATURES, LEVEL_METHOD), "damped level x raw premium")
comb_results, combined_horizon = run_folds(level_predict(COMBINED_FEATURES, LEVEL_METHOD),
                                           "combined: damped level x (slide-4 + DIST) premium")
combined_results = pd.concat([base_results, damped_raw_results, comb_results])
display(fold_table(combined_results))
combined_results.pivot(index="run", columns="fold", values="bias").loc[combined_results["run"].unique(), FOLD_NAMES]

fold,fold 1,fold 2,fold 3,fold 4,fold 5,5-fold mean,16-month
run,,,,,,,
baseline (6 raw columns),540.36,517.64,484.31,458.21,466.53,493.41,506.18
damped level x raw premium,537.12,515.49,484.11,454.83,464.31,491.17,500.11
combined: damped level x (slide-4 + DIST) premium,528.32,507.55,476.48,445.40,450.63,481.68,488.92


fold,fold 1,fold 2,fold 3,fold 4,fold 5,16-month
run,,,,,,
baseline (6 raw columns),-1.98,-2.93,-35.22,-31.35,-4.28,-50.00
damped level x raw premium,24.24,27.30,-1.18,-11.50,18.56,-27.13
combined: damped level x (slide-4 + DIST) premium,28.45,31.09,1.17,-9.71,19.98,-22.04


In [12]:
pd.concat({"baseline": base_horizon, "combined": combined_horizon}, axis=1)

baseline               combined              
                    n   rmse   bias        n   rmse   bias
months ahead                                              
1-3              8829 544.66  -7.21     8829 534.36   6.67
4-6              9347 526.12 -24.56     9347 513.00  -1.68
7-9              9089 506.24 -63.22     9089 489.42 -32.14
10-12            8086 480.11 -67.27     8086 456.61 -33.98
13-16           11820 476.74 -80.08    11820 453.62 -43.66

- **`STREET` as a native categorical works:** no errors, no unseen streets in any fold or in test, and it does
  not make any fold worse, so no fallback was needed.
- **The combined model beats the baseline on every fold:** 540.4 → 528.3, 517.6 → 507.6, 484.3 → 476.5,
  458.2 → 445.4, 466.5 → 450.6 (mean 493.4 → 481.7, −11.7); 16-month fold 506.2 → 488.9 (−17.3).
- **Each part helps on its own.** The damped level on the raw columns gives mean 491.2 and 16-month 500.1 (both
  logged on `macro-time-trend`). Adding the slide-4 columns and `DIST` to the premium takes that to 481.7 and
  488.9. `DIST` with `MONTH_INDEX` and no level (section 5) gives 485.5 and 494.9.
- **Horizon bias halves** at every distance past 6 months: −80.1 → −43.7 at 13-16 months, −63.2 → −32.1 at 7-9.
  The level is still under-forecast over 16 months on this cutoff.
- **On the short folds, the level now leans high:** bias +28.5 and +31.1 on folds 1 and 2 (baseline −2.0, −2.9).
  The damped trend still carries some of the 2021-23 climb when trained to 2023-12 or 2024-03.
- One configuration, baseline settings, no tuning. Columns and `DIST` were not selected per feature.

## 9. Fit on all of train, first submission (next step 4)

The hedonic level is fitted on 2021-01 to 2025-03 and forecast with the damped trend for 2025-04 to 2026-07
(test rows from 2025-03 use that month's fitted level). The combined premium model is trained on all of train
and predicts the 50,000 test rows in the file's own order. The test file has no ID column, so the submission is
a single `MONTHLY_RENT` column matched by row position.

In [13]:
full = pd.concat([train, test.assign(**{TARGET: np.nan})], ignore_index=True)
t_all, v_all = np.arange(len(train)), np.arange(len(train), len(full))
scale = level_scale(full, t_all, v_all, LEVEL_METHOD)
X = model_matrix(full, COMBINED_FEATURES)
y_train = train[TARGET].to_numpy()
final = lgbm(X.iloc[t_all], y_train / scale[t_all])
test_pred = final.predict(X.iloc[v_all]) * scale[v_all]

unseen_streets = test["STREET"][~test["STREET"].isin(CATEGORIES["STREET"].categories)]
print(f"test rows on a street train never had: {len(unseen_streets)} (predicted with STREET missing)")
assert len(test_pred) == len(raw_test) and np.isfinite(test_pred).all()

submission = pd.DataFrame({TARGET: test_pred})
submission.to_csv(OUT / "submission.csv", index=False)
print(f"wrote {OUT / 'submission.csv'}: {len(submission):,} rows")

test_month = test["RENT_APPROVAL_DATE"].dt.strftime("%Y-%m").to_numpy()
level_by_month = pd.Series(scale[v_all], index=test_month).groupby(level=0).first()
summary = pd.DataFrame({"level (2021-01 = 1)": level_by_month,
                        "median prediction": pd.Series(test_pred, index=test_month).groupby(level=0).median()})
print(f"level change, 2025-03 to 2026-07: {level_by_month.iloc[-1] / level_by_month.iloc[0] - 1:+.1%}")
print("actual median rent, 2025-01 to 2025-03:",
      train[train["MONTH_INDEX"] >= 48].groupby("MONTH_INDEX")[TARGET].median().to_dict())
summary.iloc[[0, 4, 8, 12, 16]]

test rows on a street train never had: 0 (predicted with STREET missing)
wrote ..\outputs\submission.csv: 50,000 rows


level change, 2025-03 to 2026-07: +4.3%
actual median rent, 2025-01 to 2025-03: {48: 3100.0, 49: 3150.0, 50: 3200.0}


,level (2021-01 = 1),median prediction
2025-03,1.56,"3,180.14"
2025-07,1.58,"3,238.89"
2025-11,1.60,"3,239.23"
2026-03,1.61,"3,267.33"
2026-07,1.63,"3,320.80"


- `outputs/submission.csv`: 50,000 rows, one `MONTHLY_RENT` column, test file order. Every test street was seen
  in train.
- **The damped trend raises the level 4.3% over the test window** (1.56 at 2025-03 to 1.63 at 2026-07, 2021-01 =
  1). Median prediction goes from 3,180 to 3,321. The last training months' actual medians are 3,100, 3,150 and
  3,200.
- Not yet compared with the leaderboard; that is the second half of next step 4.

## 10. Summary

**Reproduced from the progress report, all within 0.1 of the slides:**

- Slide 8/9 baseline: five folds 540.4, 517.6, 484.3, 458.2, 466.5 (mean 493.4); 16-month fold 506.2; bias −7.2 at
  1-3 months ahead, −80.1 at 13-16.
- Slide 11 level forecasts (plateau / climb): frozen 506.2 / 793.3, damped 500.1 / 620.0, straight line
  656.2 / 661.9, perfect 499.9 / 611.0. Caveat: a nine-cutoff check after the report
  (`08_level_robustness.ipynb`) shows the damped trend can overshoot badly (+533 bias from 2022-12).
- Slide 12 location features: 496.4 → 489.6 on `06`'s split. On the slide-8 folds, mean 493.4 → 485.5 and
  16-month 506.2 → 494.9.
- Slide 10 macro: not used, not re-run.

**Next steps covered here:**

- **3. Combine:** damped level × LightGBM premium on slide 4's columns (`STREET` as a native categorical) +
  `DIST`. Five folds 528.3, 507.6, 476.5, 445.4, 450.6 (mean 481.7 vs 493.4); 16-month 488.9 vs 506.2. Better on
  every fold.
- **4. First submission:** `outputs/submission.csv` from the combined model. Leaderboard comparison still to do.

**Still open:**

- **1.** Adopt the five folds + 16-month fold as the only scheme and log every run to `experiments.jsonl`. This
  notebook uses the scheme but does not log.
- **2.** Re-score location features one by one on these folds; keep only those that beat fold noise. Here all 13
  `DIST` columns go in together.
- **5.** Other models and tuning on the premium part.

In [14]:
print(f"runtime: {(time.time() - T0) / 60:.1f} min")

runtime: 1.0 min
